# Cheat sheet: `apply`, `lambda` and column operations

The examples use a different, made-up DataFrame (`df` with columns `a`, `b`, `text`, `date`), so you can adapt them to your exercises.

## 1. Idea of `apply`

| What | Code |
|---|---|
| Apply a function to **each value** of a column | `df["a"].apply(my_function)` |
| Apply a function to **each row** | `df.apply(my_function, axis=1)` |
| Apply a function to **each column** | `df.apply(my_function)` (same as `axis=0`) |
| Apply to selected columns only | `df[["a", "b"]].apply(my_function, axis=1)` |
| Store the result in a new column | `df["new"] = df["a"].apply(my_function)` |

Important: write `my_function` **without parentheses** inside `apply`. You give the function, you do not call it.

## 2. Write the function

| Part | Pattern |
|---|---|
| Definition | `def my_function(value):` |
| Docstring (PEP 257) | `"""Return a label for the value."""` |
| Return a result | `return something` |
| Several cases | `if ...:` then `elif ...:` then `else:` |
| Several conditions | `value >= 2 and value < 5` or `2 <= value < 5` |

Points to think about:

- With `if` / `elif` / `else`, pandas checks them **in order** and stops at the first one that is true. So the order of your conditions matters.
- Look at the limits in the statement (for example, a value that appears in two ranges). Decide which side it belongs to, and use `>` or `>=` accordingly.
- Every path of the function must `return` something.

## 3. `apply` on a row (`axis=1`)

| What | Code |
|---|---|
| The function receives a whole row | `def f(row):` |
| Read a column of the row | `row["a"]` |
| Use two columns | `row["a"] + row["b"]` |
| Call it | `df.apply(f, axis=1)` |

With `axis=1`, you must use the column **names** in the function. Without `axis=1`, `apply` on a DataFrame goes column by column, which is not what you want here.

## 4. `lambda` (one-line function)

| What | Code |
|---|---|
| Syntax | `lambda x: expression_with_x` |
| On a column | `df["a"].apply(lambda x: x + 1)` |
| On rows | `df.apply(lambda row: row["a"] + row["b"], axis=1)` |
| With a condition | `lambda x: "yes" if x > 3 else "no"` |

A `lambda` is the same as a `def`, but short. It has no name and no `return` word (the result is the expression). Use it for simple things only.

## 5. Operations between columns (no `apply`)

| What | Code |
|---|---|
| Arithmetic | `df["a"] + df["b"]`, `df["a"] - df["b"]`, `df["a"] * df["b"]`, `df["a"] / df["b"]` |
| With a number | `df["a"] * 100` |
| Combine | `df["a"] / df["b"] * 100` |
| Round | `(df["a"] / df["b"]).round(2)` |
| Store | `df["new"] = ...` |

This works on the whole column at once. It is the fastest way. `apply` gives the same result, but more slowly. The exercises ask you to compare both.

## 6. Work with text

| Goal | Code |
|---|---|
| Split a string | `"a-b-c".split("-")` gives `["a", "b", "c"]` |
| Count items in a list | `len(my_list)` |
| Split a whole column | `df["text"].str.split("-")` |
| Last element of each list | `df["text"].str.split("-").str[-1]` |
| First element of each list | `df["text"].str.split("-").str[0]` |
| Remove spaces around | `df["text"].str.strip()` |

## 7. Work with dates

| Goal | Code |
|---|---|
| Check the type | `df.dtypes` |
| Convert a column | `df["date"] = pd.to_datetime(df["date"])` |
| With a format (month/day/year) | `pd.to_datetime(df["date"], format="%m/%d/%Y")` |
| Get the year | `df["date"].dt.year` |
| Month, day | `.dt.month`, `.dt.day` |
| Another way for text dates | split on the separator, then take one part |

The `.dt` accessor only works on a datetime column. Convert first.

## 8. Count and list values

| Goal | Code |
|---|---|
| List the different values | `df["col"].unique()` |
| Number of different values | `df["col"].nunique()` |
| Count of each value | `df["col"].value_counts()` |
| Sort values | `sorted(df["col"].unique())` |

## 9. Tools by mission

| Mission | Tools to look at |
|---|---|
| 1 | function with `if` / `elif` / `else`, `apply` on a column |
| 2 | `.split(...)`, `len`, function, `apply` on a column |
| 3 | same idea as 2, with a `lambda` |
| 4 | operations between columns (part 5) |
| 5 | function on a row, `apply(..., axis=1)`; then on the whole DataFrame and on selected columns |
| 6 | same as 5, with a `lambda` |
| 7 | part 6 or part 7, then part 8 |

## 10. Common mistakes

| Error | Cause | Fix |
|---|---|---|
| `SyntaxError` after `=` | The right side is empty | Write the `apply` call |
| Column has the wrong name | The statement says one name, your code uses another | Copy the name from the statement, exactly |
| `TypeError: 'function' object ...` | `apply(f())` with parentheses | Write `apply(f)` |
| `KeyError` | Wrong column name, or spaces in it | Check `df.columns` |
| Result is `None` | The function has no `return` for some case | Add a `return` in every branch |
| `apply` runs but all values are wrong | Missing `axis=1` for rows | Add `axis=1` |
| `.dt` or `.str` error | Wrong type | Check `df.dtypes` |
| Division gives a surprise | Zero in the denominator, or percent not multiplied by 100 | Check the formula in the statement |

## 11. Tip about your data

In your `books.info()`, the column `num_pages` is shown as `"  num_pages"` with spaces at the start. This can cause a `KeyError` later. You can clean all the names with:

```python
books.columns = books.columns.str.strip()
```

Then check with `books.columns`.

## 12. PEP 8 and PEP 257 reminders

- One-line docstring in triple double quotes, no spaces inside the quotes, ending with a period.
- Function names in `snake_case`, in lowercase with underscores.
- No spaces around `=` in keyword arguments: `nrows=13`, not `nrows = 13`.
- Two spaces before an inline comment, and a space after the `#`.
- Lines under 79 characters.

Write your first mission and send it to me when you want a check.

In [1]:
import pandas as pd

### Welcome to the Apply Challenge

In [2]:
link = 'https://raw.githubusercontent.com/chriszapp/datasets/main/books.csv'
books = pd.read_csv(link, nrows = 13)
books.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [6]:
# Firstly, have a look at the dataset
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              13 non-null     int64  
 1   title               13 non-null     str    
 2   authors             13 non-null     str    
 3   average_rating      13 non-null     float64
 4   isbn                13 non-null     str    
 5   isbn13              13 non-null     int64  
 6   language_code       13 non-null     str    
 7     num_pages         13 non-null     int64  
 8   ratings_count       13 non-null     int64  
 9   text_reviews_count  13 non-null     int64  
 10  publication_date    13 non-null     str    
 11  publisher           13 non-null     str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.3 KB


In [7]:
books.shape

(13, 12)

In [13]:
# Mission 1: use apply to create a new column called 'reviews_opinion': if the ratings is between 4.5 and 5, the reviews_opinion is 'very good';
# if the rating is between 4 and 4.5, the reviews_opinion is 'good'; if the rating is between 3 and 4, the reviews_opinions will be 'neutral';
# if the rating is up to 3 the reviews_opinion is going to be 'bad'

def get_opinion(rating):
    """Return a text opinion for an average rating."""
    if rating <= 3:
        return "bad"
    elif rating <= 4:
        return "neutral"
    elif rating <= 4.5:
        return "good"
    else:
        return "very good"


books["reviews_opinion"] = books["average_rating"].apply(get_opinion)
books.head()


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good


In [17]:
# Mission 2: using apply with a function, create a column with the number of authors per book
# (note that in the column 'authors', each name of the author is divided with the character '/')

# Add your code here
def author_count(x):
    return len(x.split("/"))

books["author_count"] = books["authors"].apply(author_count)
books.head()


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good,2
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good,2
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good,1
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good,2
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good,2


In [18]:
# Mission 3: rewrite the same task as Mission 2 using the lamdba function

# Add your code here

books["author_count_v2"] = books["authors"].apply(lambda x: len(x.split("/")))
books.head()


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count,author_count_v2
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good,2,2
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good,2,2
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good,1,1
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good,2,2
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good,2,2


In [23]:
# Mission 4: create a column 'comment_ration'.
# Use the columns 'ratings_count' and 'text_reviews_count' to calculate the percentage of how many text review per reviews are per book:
# percentage = text / reviews * 100
# Use the mathematical operations between columns

# Add your code here
books["comment_ration"] = round(((books["text_reviews_count"] / books["ratings_count"]) * 100), 2)
books



,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count,author_count_v2,comment_ration
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good,2,2,1.32
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good,2,2,1.36
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good,1,1,3.85
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good,2,2,1.55
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good,2,2,0.40
5,9,"Unauthorized Harry Potter Book Seven News: ""Ha...",W. Frederick Zimmerman,3.74,0976540606,9780976540601,en-US,152,19,1,4/26/2005,Nimble Books,neutral,neutral,1,1,5.26
6,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,9780439827607,eng,3342,28242,808,9/12/2005,Scholastic,very good,very good,1,1,2.86
7,12,The Ultimate Hitchhiker's Guide: Five Complete...,Douglas Adams,4.38,0517226952,9780517226957,eng,815,3628,254,11/1/2005,Gramercy Books,good,good,1,1,7.00
8,13,The Ultimate Hitchhiker's Guide to the Galaxy ...,Douglas Adams,4.38,0345453743,9780345453747,eng,815,249558,4080,4/30/2002,Del Rey Books,good,good,1,1,1.63
9,14,The Hitchhiker's Guide to the Galaxy (Hitchhik...,Douglas Adams,4.22,1400052920,9781400052929,eng,215,4930,460,8/3/2004,Crown,good,good,1,1,9.33


In [37]:
# Mission 5: obtain the same result of Mission 4 using apply with a function.
# Try both methods to apply to the whole dataset and apply to selected columns

def get_comment_ratio(row):
    """Return the percentage of text reviews per rating."""
    return round(row["text_reviews_count"] / row["ratings_count"] * 100, 2)

# Add your code here
books["comment_ration_v2"] = books[["text_reviews_count", "ratings_count"]].apply(get_comment_ratio, axis=1)
books.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count,author_count_v2,comment_ration,comment_ration_v2
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good,2,2,1.32,1.32
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good,2,2,1.36,1.36
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good,1,1,3.85,3.85
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good,2,2,1.55,1.55
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good,2,2,0.40,0.40


In [38]:
# Mission 6: obtain the same result of Mission 5 using apply lambda
# Try both methods to apply to the whole dataset and apply to selected columns

# Add your code here

books["comment_ration_v3"] = books[["text_reviews_count", "ratings_count"]].apply(lambda row: round(row["text_reviews_count"] / row["ratings_count"] * 100, 2), axis=1)
books

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count,author_count_v2,comment_ration,comment_ration_v2,comment_ration_v3
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,very good,very good,2,2,1.32,1.32,1.32
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,good,good,2,2,1.36,1.36,1.36
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,good,good,1,1,3.85,3.85,3.85
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,very good,very good,2,2,1.55,1.55,1.55
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,very good,very good,2,2,0.40,0.40,0.40
5,9,"Unauthorized Harry Potter Book Seven News: ""Ha...",W. Frederick Zimmerman,3.74,0976540606,9780976540601,en-US,152,19,1,4/26/2005,Nimble Books,neutral,neutral,1,1,5.26,5.26,5.26
6,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,9780439827607,eng,3342,28242,808,9/12/2005,Scholastic,very good,very good,1,1,2.86,2.86,2.86
7,12,The Ultimate Hitchhiker's Guide: Five Complete...,Douglas Adams,4.38,0517226952,9780517226957,eng,815,3628,254,11/1/2005,Gramercy Books,good,good,1,1,7.00,7.00,7.00
8,13,The Ultimate Hitchhiker's Guide to the Galaxy ...,Douglas Adams,4.38,0345453743,9780345453747,eng,815,249558,4080,4/30/2002,Del Rey Books,good,good,1,1,1.63,1.63,1.63
9,14,The Hitchhiker's Guide to the Galaxy (Hitchhik...,Douglas Adams,4.22,1400052920,9781400052929,eng,215,4930,460,8/3/2004,Crown,good,good,1,1,9.33,9.33,9.33


In [44]:
books["publication_date"].info

<bound method Series.info of 0     9/16/2006
1      9/1/2004
2     11/1/2003
3      5/1/2004
4     9/13/2004
5     4/26/2005
6     9/12/2005
7     11/1/2005
8     4/30/2002
9      8/3/2004
10    3/23/2005
11    1/17/1996
12    9/14/2004
Name: publication_date, dtype: str>

In [47]:
# Mission 7: extract the year from the publication_date column. How many years of publications are recorded in this dataset? Which are them?
# Add your code here

books["publication_date"] = pd.to_datetime(books["publication_date"], format="%m/%d/%Y")
books.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,review_opinion,reviews_opinion,author_count,author_count_v2,comment_ration,comment_ration_v2,comment_ration_v3
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,2006-09-16,Scholastic Inc.,very good,very good,2,2,1.32,1.32,1.32
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,2004-09-01,Scholastic Inc.,good,good,2,2,1.36,1.36,1.36
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,2003-11-01,Scholastic,good,good,1,1,3.85,3.85,3.85
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,2004-05-01,Scholastic Inc.,very good,very good,2,2,1.55,1.55,1.55
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,2004-09-13,Scholastic,very good,very good,2,2,0.40,0.40,0.40


In [48]:
books["publication_date"].info

<bound method Series.info of 0    2006-09-16
1    2004-09-01
2    2003-11-01
3    2004-05-01
4    2004-09-13
5    2005-04-26
6    2005-09-12
7    2005-11-01
8    2002-04-30
9    2004-08-03
10   2005-03-23
11   1996-01-17
12   2004-09-14
Name: publication_date, dtype: datetime64[us]>

In [54]:
books["publication_date"].dt.year.value_counts().count()

np.int64(6)

In [55]:
books["publication_date"].dt.year.value_counts()

publication_date
2004    5
2005    4
2006    1
2003    1
2002    1
1996    1
Name: count, dtype: int64